##**Step 1: Install Libraries**

*   requests - send requests to website
*   beautifulsoup4 - read & parse HTML
* pandas - store data in table format


In [1]:
!pip install requests
!pip install beautifulsoup4
!pip install pandas

##**Step 2: Import Libraries**


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

##**Step 3: Define Website Endpoint**

In [3]:
url = "https://soyacincau.com/?ajax-request=jnews"

##**Step 4: Create Payload**

In [4]:
payload = {
    "ajax-request": "jnews",
    "lang": "en_US",
    "action": "jnews_module_ajax_jnews_block_23",
    "module": "true",
    "data[current_page]": 1,
    "data[attribute][number_post]": 6
}

##**Step 5: Request Website**



In [5]:
response = requests.post(url, data=payload)

##**Step 6: Check Status (Website Connection)**


Before crawling thousands of articles, we need to confirm that:

* The website can be accessed by our program.
* The URL is valid.
* The server returns the webpage successfully.
* Status code **200** indicates webpage was successfully retrieved and is ready for processing.




In [6]:
print(response.status_code)

200


##**Step 6: Convert HTML into Soup**

In [7]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

##**Step 7:Inspect Returned Data**


In [8]:
print(response.text[:1000])

{"content":"<div class=\"jeg_posts_wrap\">\n                    <div class=\"jeg_posts jeg_load_more_flag\"> \n                        <article class=\"jeg_post jeg_pl_md_1 format-standard\">\n                    <div class=\"jeg_thumb\">\n                        \n                        <a href=\"https:\/\/soyacincau.com\/2026\/06\/17\/cap-free-illegal-video-streaming-exposing-you-to-malware-and-fraud\/\"><div class=\"thumbnail-container  size-715 \"><img width=\"350\" height=\"250\" src=\"https:\/\/soyacincau.com\/wp-content\/uploads\/2026\/06\/260617-video-streaming-remote-unsplash-pinho-350x250.jpg\" class=\"attachment-jnews-350x250 size-jnews-350x250 wp-post-image\" alt=\"\" decoding=\"async\" srcset=\"https:\/\/soyacincau.com\/wp-content\/uploads\/2026\/06\/260617-video-streaming-remote-unsplash-pinho-350x250.jpg 350w, https:\/\/soyacincau.com\/wp-content\/uploads\/2026\/06\/260617-video-streaming-remote-unsplash-pinho-120x86.jpg 120w, https:\/\/soyacincau.com\/wp-content\/uploa

##**Step 8: Convert Response to JSON**

In [9]:
response.text

data = response.json()
print(data.keys())

dict_keys(['content', 'next', 'prev'])


##**Step 9: Extract the html from JSON**


In [10]:
html = data["content"]

print(html[:500])

<div class="jeg_posts_wrap">
                    <div class="jeg_posts jeg_load_more_flag"> 
                        <article class="jeg_post jeg_pl_md_1 format-standard">
                    <div class="jeg_thumb">
                        
                        <a href="https://soyacincau.com/2026/06/17/cap-free-illegal-video-streaming-exposing-you-to-malware-and-fraud/"><div class="thumbnail-container  size-715 "><img width="350" height="250" src="https://soyacincau.com/wp-content/uploads/20


##**Step 10: Create BeautofulSoup Object**


In [11]:
soup = BeautifulSoup(html, "html.parser")
print(type(soup))

<class 'bs4.BeautifulSoup'>


##**Step 11: Find All Articles**

In [12]:
articles = soup.find_all("article")
print(len(articles))

6


##**Step 12: Inspect One Article**

In [13]:
print(articles[0].prettify()[:3000])

<article class="jeg_post jeg_pl_md_1 format-standard">
 <div class="jeg_thumb">
  <a href="https://soyacincau.com/2026/06/17/cap-free-illegal-video-streaming-exposing-you-to-malware-and-fraud/">
   <div class="thumbnail-container size-715">
    <img alt="" class="attachment-jnews-350x250 size-jnews-350x250 wp-post-image" decoding="async" height="250" sizes="(max-width: 350px) 100vw, 350px" src="https://soyacincau.com/wp-content/uploads/2026/06/260617-video-streaming-remote-unsplash-pinho-350x250.jpg" srcset="https://soyacincau.com/wp-content/uploads/2026/06/260617-video-streaming-remote-unsplash-pinho-350x250.jpg 350w, https://soyacincau.com/wp-content/uploads/2026/06/260617-video-streaming-remote-unsplash-pinho-120x86.jpg 120w, https://soyacincau.com/wp-content/uploads/2026/06/260617-video-streaming-remote-unsplash-pinho-750x536.jpg 750w, https://soyacincau.com/wp-content/uploads/2026/06/260617-video-streaming-remote-unsplash-pinho-1140x815.jpg 1140w" width="350"/>
   </div>
  </a>
  

##**Step 13: Extract One Article**

In [14]:
article = articles[0]

title = article.find("h3", class_="jeg_post_title").get_text(strip=True)

url = article.find( "h3", class_="jeg_post_title").find("a")["href"]

category = article.find("div", class_="jeg_post_category").get_text(strip=True)

date = article.find("div", class_="jeg_meta_date").get_text(strip=True)

excerpt = article.find("div", class_="jeg_post_excerpt").get_text(strip=True)

print("Title:", title)
print("URL:", url)
print("Category:", category)
print("Date:", date)
print("Excerpt:", excerpt)

Title: Your “free” streaming service could be exposing you to malware and fraud
URL: https://soyacincau.com/2026/06/17/cap-free-illegal-video-streaming-exposing-you-to-malware-and-fraud/
Category: Digital Life
Date: June 17, 2026
Excerpt: A study has found that millions consumers using illegal streaming services across Asia-Pacific could be exposing themselves to significant cybersecurity, ...


##**Step 14: Extract All Articles on Current Page**

In [15]:
all_articles = []

for article in articles:

    title = article.find("h3", class_="jeg_post_title").get_text(strip=True)

    url = article.find("h3", class_="jeg_post_title").find("a")["href"]

    category = article.find("div", class_="jeg_post_category").get_text(strip=True)

    excerpt = article.find("div", class_="jeg_post_excerpt").get_text(strip=True)

    date = article.find("div", class_="jeg_meta_date").get_text(strip=True)

    all_articles.append({
        "title": title,
        "url": url,
        "category": category,
        "date": date,
        "excerpt": excerpt
    })

print("Articles found:", len(all_articles))
print(all_articles[0])

Articles found: 6
{'title': 'Your “free” streaming service could be exposing you to malware and fraud', 'url': 'https://soyacincau.com/2026/06/17/cap-free-illegal-video-streaming-exposing-you-to-malware-and-fraud/', 'category': 'Digital Life', 'date': 'June 17, 2026', 'excerpt': 'A study has found that millions consumers using illegal streaming services across Asia-Pacific could be exposing themselves to significant cybersecurity, ...'}


##**Step 15: Save to JSON**

In [16]:
import json

with open("raw_data.json", "w", encoding="utf-8") as f:
    json.dump(all_articles, f, ensure_ascii=False, indent=4)

print("Saved!", len(all_articles), "records")

Saved! 6 records


In [17]:
import os
print(os.path.exists("raw_data.json"))

True


##**Step 16: AJAX Request**


In [18]:
import requests

url = "https://soyacincau.com/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://soyacincau.com",
    "Referer": "https://soyacincau.com/"
}

In [19]:
payload = {
    "ajax-request": "jnews",
    "lang": "en_US",
    "action": "jnews_module_ajax_jnews_block_23",
    "module": "true",
    "data[current_page]": 1,
    "data[attribute][number_post]": 6,
    "data[attribute][post_offset]": 0,
    "data[attribute][post_type]": "post",
    "data[attribute][content_type]": "all",
    "data[attribute][sort_by]": "latest",
    "data[attribute][pagination_mode]": "loadmore"
}

session = requests.Session()

response = session.post(url, data=payload, headers=headers)

print("Status:", response.status_code)
print(response.text[:300])

Status: 200
{"content":"<div class=\"jeg_posts_wrap\">\n                    <div class=\"jeg_posts jeg_load_more_flag\"> \n                        <article class=\"jeg_post jeg_pl_md_1 format-standard\">\n                    <div class=\"jeg_thumb\">\n                        \n                        <a href=\"


##**Step 17: Create Extraction Function**

In [20]:
def extract_articles(html):

    soup = BeautifulSoup(html, "html.parser")

    articles = soup.find_all("article")

    records = []

    for article in articles:

        title = article.find(
            "h3",
            class_="jeg_post_title"
        ).get_text(strip=True)

        url = article.find(
            "h3",
            class_="jeg_post_title"
        ).find("a")["href"]

        category = article.find(
            "div",
            class_="jeg_post_category"
        ).get_text(strip=True)

        date = article.find(
            "div",
            class_="jeg_meta_date"
        ).get_text(strip=True)

        excerpt = article.find(
            "div",
            class_="jeg_post_excerpt"
        ).get_text(strip=True)

        records.append({
            "title": title,
            "url": url,
            "category": category,
            "date": date,
            "excerpt": excerpt
        })

    return records

##**Step 18: Crawl Multiple Pages**

In [21]:
all_records = []

for page in range(1, 11):

    print(f"Scraping page {page}")

    payload["data[current_page]"] = page

    response = session.post(
        url,
        data=payload,
        headers=headers
    )

    data = response.json()

    html = data["content"]

    records = extract_articles(html)

    print("Found", len(records), "articles")

    all_records.extend(records)

Scraping page 1
Found 6 articles
Scraping page 2
Found 6 articles
Scraping page 3
Found 6 articles
Scraping page 4
Found 6 articles
Scraping page 5
Found 6 articles
Scraping page 6
Found 6 articles
Scraping page 7
Found 6 articles
Scraping page 8
Found 6 articles
Scraping page 9
Found 6 articles
Scraping page 10
Found 6 articles


# Step added by phase2 member to get the actual 100k records

In [ ]:
import time
import json
from google.colab import drive
drive.mount("/content/drive")

all_records = []

for page in range(1, 20000):

    print(f"Scraping page {page}")

    payload["data[current_page]"] = page

    response = session.post(
        url,
        data=payload,
        headers=headers
    )

    data = response.json()

    html = data["content"]

    records = extract_articles(html)

    if not records:
        print(f"No more articles found at page {page}, stopping.")
        break

    print("Found", len(records), "articles")

    all_records.extend(records)

    if len(all_records) % 1000 == 0:
        with open("/content/drive/MyDrive/raw_data.json", "w", encoding="utf-8") as f:
            json.dump(all_records, f, ensure_ascii=False, indent=4)
        print(f"Auto-saved at {len(all_records)} records")

    if len(all_records) >= 100000:
        print("Reached 100k records, stopping.")
        break

    time.sleep(0.5)

print("Total records collected:", len(all_records))

ValueError: mount failed

##**Step 19: Check Results**

In [ ]:
print("Total records:", len(all_records))

print(all_records[0])

##**Step 20: Save Everything to JSON**

In [ ]:
import json

with open("/content/drive/MyDrive/raw_data.json", "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=4)

print("Saved", len(all_records), "records to Drive")

##**Step 21: Progressive Saving**

In [ ]:
import json

all_records = []

for page in range(1, 11):

    print(f"Scraping page {page}")

    payload["data[current_page]"] = page

    response = session.post(
        url,
        data=payload,
        headers=headers
    )

    data = response.json()

    html = data["content"]

    records = extract_articles(html)

    all_records.extend(records)

    with open(
        "raw_data.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            all_records,
            f,
            ensure_ascii=False,
            indent=4
        )

    print(
        f"Saved {len(all_records)} records"
    )

##**Step 22: Verify JSON File**

In [ ]:
import json

with open("raw_data.json", encoding="utf-8") as f:

    data = json.load(f)

print("Total records:", len(data))
print(data[0])

##**Step 23: Check Duplicate URLs**

In [ ]:
urls = [record["url"] for record in all_records]

print("Total URLs:", len(urls))
print("Unique URLs:", len(set(urls)))

##**Step 24: Save CSV Version**

In [ ]:
import pandas as pd

df = pd.DataFrame(all_records)

df.to_csv(
    "raw_data.csv",
    index=False,
    encoding="utf-8"
)

print(df.head())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# then save to:
with open("/content/drive/MyDrive/raw_data.json", "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=4)